In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ! pip install byaldi
# ! pip install -U torchao

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 49.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 49.6 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 66.5 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 88.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 75.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 191.7 MB/s  0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.8.0
    Uninstalling triton-3.8.0:
      Successfully uninstalled triton-3.8.0
  Attempting uninstall: nvidia-nccl-cu13
    Found existing installation: nvidia-nccl-cu13 2.30.7
    Uninstalling nvidia-nccl-cu13-2.30.7:
      Successfully uninstalled nvidia-nccl-cu13-2.30.7
  Attempting uninstall: nvidia-cudnn-cu1

In [48]:
# from sentence_transformers import MultiVectorEncoder
from byaldi import RAGMultiModalModel
from datasets import load_dataset
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import torch
import time
import glob
import os

# Folder Paths

In [49]:
ROOT = "/content/drive/MyDrive/ColPali IR project"
PDF_PATH = ROOT + "/Data/Arxive PDFs"
IMG_PATH = ROOT + "/Data/Pages/Images"
TXT_PATH = ROOT + "/Data/Pages/Texts"
QRS_PATH = ROOT + "/Data/Queries/colpali_50_queries.csv"

In [ ]:

def flatten_images_folder(root_path, output_folder="All_Images"):
    output_path = os.path.join(root_path, output_folder)
    os.makedirs(output_path, exist_ok=True)

    image_paths = glob.glob(
        os.path.join(root_path, "**", "*.png"),
        recursive=True
    )

    for src in image_paths:
        if os.path.dirname(src) == output_path:
            continue
        shutil.copy2(src, os.path.join(output_path, os.path.basename(src)))

    print(f"{len(image_paths)} images copied to {output_path}")
    return output_path

fll

# ColQwen Inference on ArxiveQA and docQA

In [34]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

Device : cuda


# Queries dataset

In [35]:
queries = pd.read_csv(QRS_PATH)
queries.head()

,questions,paper,page_no,question_type
0,What role does dropout play in the unsupervise...,paper_01.pdf,2,text
1,Which dropout probability gives the strongest ...,paper_01.pdf,4,table
2,Why are entailment pairs from SNLI and MNLI us...,paper_01.pdf,5,text
3,How does first-last average pooling compare wi...,paper_01.pdf,8,table
4,Why does SPLADE retain advantages of tradition...,paper_02.pdf,2,text


# DocVQA and ArxivQA Indexing

In [38]:
# ColQwen MultiVector
model = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0",
    device=device
)

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [39]:
# paper_05 (20 pages) timed Indexing
start = time.time()

model.index(
        input_path=IMG_PATH + "/paper_05",
        index_name="20_pages_indexing",
        store_collection_with_index=True,
        overwrite=True
    )

end = time.time()
print(f"Indexing time for 20 pages: {end - start}")

overwrite is on. Deleting existing index 20_pages_indexing to build a new one.
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/paper_05/paper_05_page_011.png
Added page 1 of document 0 to index.
Index exported to .byaldi/20_pages_indexing
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/paper_05/paper_05_page_012.png
Added page 1 of document 1 to index.
Index exported to .byaldi/20_pages_indexing
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/paper_05/paper_05_page_013.png
Added page 1 of document 2 to index.
Index exported to .byaldi/20_pages_indexing
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/paper_05/paper_05_page_014.png
Added page 1 of document 3 to index.
Index exported to .byaldi/20_pages_indexing
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/Pages/Images/paper_05/paper_05_page_015.png
Added page 1 of document 4 to index.
Index exported to .byaldi/20_p

In [47]:
# Finale Model for the hole 300 pages
model = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0",
    device=device
)

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [ ]:

# Extraction de tous les fichiers PNG de tous les sous-dossiers
image_paths = sorted(
    glob.glob(
        os.path.join(IMG_PATH, "**", "*.png"),
        recursive=True
    )
)

start = time.time()

# Indexation image par image pour contourner les limitations de structure de Byaldi
for i, image_path in enumerate(image_paths):
    # On écrase l'index uniquement pour la première image, puis on ajoute les autres
    is_first = (i == 0)
    model.index(
        input_path=image_path,
        index_name="300_pages_index",
        store_collection_with_index=False,
        overwrite=is_first
    )

end = time.time()

print(f"Indexed {len(image_paths)} pages")
print(f"Indexing time: {end - start:.2f} seconds")

# Retrieval

In [ ]:
# Retrieval function

def retrieval(df, model, k=5):

    all_retrieved = []
    all_scores = []

    for idx, row in df.iterrows():

        query_text = row["query"]
        results = model.search(query_text, k=k)

        retrieved = [r["doc_id"] for r in results]
        scores = [r["score"] for r in results]

        all_retrieved.append(retrieved)
        all_scores.append(scores)

    return np.array(all_retrieved), all_scores

In [ ]:
# top-5 retrieved documents and their scores
all_retrieved_docvqa, all_scores_docvqa = retrieval(docvqa_queries, model_docvqa, k=5)

In [ ]:
# top-5 retrieved documents and their scores
all_retrieved_arxivqa, all_scores_docvqa = retrieval(arxivqa_queries, model_arxivqa, k=5)

# Evaluation

In [ ]:
# Compute Discounted Cumulative Gain (DCG) and normalized DCG (nDCG) at K

def dcg_at_k(rels, k=5):

    rels = np.array(rels[:k], dtype=float)
    return float(np.sum((2**rels - 1) / np.log2(np.arange(1, len(rels)+1) + 1)))

# Compute nDCG@K assuming a single relevant ground-truth document target
def ndcg_at_k(retrieved, relevant, k=5):

    rels = [1 if d in relevant else 0 for d in retrieved[:k]]
    dcg = dcg_at_k(rels, k)

    ideal_rels = [1] + [0] * (k - 1)
    idcg = dcg_at_k(ideal_rels, k)

    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
# Compute nDCG@5
relevant = np.arange(len(arxivqa_queries))
ndcg5 = np.mean([ndcg_at_k(retrieved, relevant, k=5) for retrieved, relevant in zip(all_retrieved_docvqa, [[r] for r in relevant])])
print(f"DocVQA nDCG@5 : {ndcg5}")

DocVQA nDCG@5 : 0.5486104402650461


In [ ]:
# Compute nDCG@5
ndcg5_arxivqa = np.mean([ndcg_at_k(retrieved, relevant, k=5) for retrieved, relevant in zip(all_retrieved_arxivqa, [[r] for r in relevant])])
print(f"ArxivQA nDCG@5 : {ndcg5_arxivqa}")

ArxivQA nDCG@5 : 0.8763568400622681
